In [1]:
# =========================================
# 0. CONFIG
# =========================================
import os
os.environ["PAGER"] = "cat"

# Credenciais via variáveis de ambiente (com fallback para desenvolvimento)
PG_HOST     = os.getenv("PG_HOST",     "localhost")
PG_DB       = os.getenv("PG_DB",       "atividade2")
PG_USER     = os.getenv("PG_USER",     "postgres")
PG_PASSWORD = os.getenv("PG_PASSWORD", "postgres")

DUMP_ORIGINAL = "backup_atividade_2.sql"
DUMP_LIMPO    = "backup_limpo.sql"
MAX_NEW_TOKENS = 512   # aumentado para evitar truncamento do [RESULT]
BATCH_LIMIT    = 10    # None = avaliar tudo; coloque um int para testar
REFAZER_AVALIACAO = True  # True = apaga avaliações existentes e refaz tudo
PROMPT_VERSION = "prometheus-v1"

RUBRICA_OBJETIVA = (
    "Nota 1: A resposta indica uma alternativa diferente do gabarito (errou a questão).\n"
    "Nota 5: A resposta indica exatamente a alternativa do gabarito (acertou a questão)."
)

RUBRICA_DISCURSIVA = (
    "Nota 1: Resposta incorreta; cita leis inexistentes ou confunde institutos jurídicos básicos.\n"
    "Nota 2: Conclusão correta, mas a fundamentação é vaga ou cita artigos de lei errados.\n"
    "Nota 3: Resposta correta e bem fundamentada, mas falta clareza ou omite detalhes importantes.\n"
    "Nota 4: Resposta excelente, alinhada ao gabarito, com fundamentação legal precisa.\n"
    "Nota 5: Resposta excepcional — fundamentada, cita jurisprudência relevante (STF/STJ) "
    "e demonstra raciocínio jurídico de alto nível."
)


In [2]:
# =========================================
# 1. INSTALAR DEPENDÊNCIAS
# =========================================
!apt-get -y install postgresql postgresql-contrib > /dev/null
!pip install -q psycopg2-binary transformers accelerate bitsandbytes sentencepiece

In [3]:
# =========================================
# 2. INICIAR POSTGRES
# =========================================
!service postgresql start

 * Starting PostgreSQL 14 database server
   ...done.


In [4]:
# =========================================
# 3. CONFIGURAR USUÁRIO
# =========================================
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD '{PG_PASSWORD}';"

ALTER ROLE


In [5]:
# =========================================
# 4. CRIAR BANCO
# =========================================
!sudo -u postgres createdb {PG_DB} || true

createdb: error: database creation failed: ERROR:  database "atividade2" already exists


In [6]:
# =========================================
# 5. LIMPAR DUMP PARA COMPATIBILIDADE LOCAL
# =========================================
if not os.path.exists(DUMP_ORIGINAL):
    raise FileNotFoundError(f"❌ {DUMP_ORIGINAL} não encontrado")

linhas_limpas = []
with open(DUMP_ORIGINAL, encoding="utf-8") as f:
    for linha in f:
        if linha.startswith("\restrict") or linha.startswith("\unrestrict"):
            continue
        if "transaction_timeout" in linha:
            continue
        if "OWNER TO diegobispo" in linha:
            continue
        linhas_limpas.append(linha)

with open(DUMP_LIMPO, "w", encoding="utf-8") as f:
    f.writelines(linhas_limpas)

print(f"✅ Dump limpo criado ({len(linhas_limpas)} linhas)")


✅ Dump limpo criado (3275 linhas)


In [7]:
# =========================================
# 6. RESTORE
# =========================================
import subprocess

restore_cmd = [
    "psql",
    "-h", PG_HOST,
    "-U", PG_USER,
    "-d", PG_DB,
    "-v", "ON_ERROR_STOP=1",
    "-f", DUMP_LIMPO,
]
restore_env = {**os.environ, "PGPASSWORD": PG_PASSWORD}
restore_result = subprocess.run(
    restore_cmd,
    env=restore_env,
    text=True,
    capture_output=True,
)

if restore_result.returncode != 0:
    print(restore_result.stdout[-4000:])
    print(restore_result.stderr[-4000:])
    raise RuntimeError(f"❌ Falha ao restaurar {DUMP_LIMPO}")

print("✅ Banco restaurado sem erros")


psql:backup_limpo.sql:39: ERROR:  relation "avaliacoes_juiz" already exists
psql:backup_limpo.sql:42: ERROR:  role "diegobispo" does not exist
psql:backup_limpo.sql:54: ERROR:  relation "avaliacoes_juiz_id_avaliacao_seq" already exists
psql:backup_limpo.sql:57: ERROR:  role "diegobispo" does not exist
psql:backup_limpo.sql:74: ERROR:  relation "datasets" already exists
psql:backup_limpo.sql:77: ERROR:  role "diegobispo" does not exist
psql:backup_limpo.sql:89: ERROR:  relation "datasets_id_dataset_seq" already exists
psql:backup_limpo.sql:92: ERROR:  role "diegobispo" does not exist
psql:backup_limpo.sql:112: ERROR:  relation "modelos" already exists
psql:backup_limpo.sql:115: ERROR:  role "diegobispo" does not exist
psql:backup_limpo.sql:127: ERROR:  relation "modelos_id_modelo_seq" already exists
psql:backup_limpo.sql:130: ERROR:  role "diegobispo" does not exist
psql:backup_limpo.sql:149: ERROR:  relation "perguntas" already exists
psql:backup_limpo.sql:152: ERROR:  role "diegobispo

In [8]:
# =========================================
# 7. TESTE DE CONEXÃO
# =========================================
!PGPASSWORD={PG_PASSWORD} psql -h {PG_HOST} -U {PG_USER} -d {PG_DB} -c "SELECT COUNT(*) FROM perguntas;" -P pager=off

 count 
-------
  2420
(1 row)



In [ ]:
# =========================================
# 7A. VALIDAR EXTENSAO MINIMA DO SCHEMA 2+1
# =========================================
import subprocess

schema_validation_sql = r"""
WITH expected_tables(table_name) AS (
    VALUES
        ('respostas_auxiliares_juiz'),
        ('avaliacao_erros'),
        ('human_audits'),
        ('decisoes_finais_julgamento')
), expected_columns(column_name) AS (
    VALUES
        ('papel_juiz'),
        ('rodada_julgamento'),
        ('motivo_acionamento'),
        ('id_resposta_auxiliar')
), expected_constraints(conname) AS (
    VALUES
        ('avaliacoes_juiz_papel_juiz_check'),
        ('avaliacoes_juiz_rodada_julgamento_check'),
        ('respostas_auxiliares_juiz_objetivo_check'),
        ('avaliacao_erros_severidade_check'),
        ('decisoes_finais_julgamento_metodo_decisao_check'),
        ('respostas_auxiliares_juiz_id_pergunta_fkey'),
        ('respostas_auxiliares_juiz_id_modelo_juiz_fkey'),
        ('avaliacoes_juiz_id_resposta_auxiliar_fkey'),
        ('avaliacao_erros_id_avaliacao_fkey'),
        ('human_audits_id_avaliacao_fkey'),
        ('decisoes_finais_julgamento_id_resposta_ativa1_fkey')
), missing_tables AS (
    SELECT table_name FROM expected_tables
    EXCEPT
    SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'
), missing_columns AS (
    SELECT column_name FROM expected_columns
    EXCEPT
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = 'avaliacoes_juiz'
), missing_constraints AS (
    SELECT conname FROM expected_constraints
    EXCEPT
    SELECT conname FROM pg_constraint
), missing AS (
    SELECT 'table' AS object_type, table_name AS object_name FROM missing_tables
    UNION ALL
    SELECT 'avaliacoes_juiz_column', column_name FROM missing_columns
    UNION ALL
    SELECT 'constraint', conname FROM missing_constraints
)
SELECT * FROM missing ORDER BY object_type, object_name;
"""

validation_cmd = [
    "psql",
    "-h", PG_HOST,
    "-U", PG_USER,
    "-d", PG_DB,
    "-v", "ON_ERROR_STOP=1",
    "-c", schema_validation_sql,
    "-P", "pager=off",
]
validation_result = subprocess.run(
    validation_cmd,
    env={**os.environ, "PGPASSWORD": PG_PASSWORD},
    text=True,
    capture_output=True,
)

print(validation_result.stdout)
if validation_result.returncode != 0:
    print(validation_result.stderr)
    raise RuntimeError("❌ Falha na validacao do schema 2+1")
if "(0 rows)" not in validation_result.stdout:
    raise RuntimeError("❌ Schema 2+1 incompleto; verifique os objetos ausentes acima")

print("✅ Extensao minima do schema 2+1 validada")


In [9]:
# =========================================
# 8. CONEXÃO PYTHON
# =========================================
import psycopg2
from psycopg2.extras import Json

conn = psycopg2.connect(
    host=PG_HOST,
    database=PG_DB,
    user=PG_USER,
    password=PG_PASSWORD
)
conn.autocommit = False  # controle explícito de transação
cur = conn.cursor()
print("✅ Conexão estabelecida")


✅ Conexão estabelecida


In [ ]:
# =========================================
# 8A. AMOSTRA TRANSACIONAL 2+1 (ROLLBACK)
# =========================================
# Valida FKs e inserts minimos sem poluir o banco restaurado.

try:
    cur.execute("BEGIN;")

    cur.execute("""
        SELECT r.id_resposta, r.id_pergunta
        FROM respostas_atividade_1 r
        ORDER BY r.id_resposta
        LIMIT 1;
    """)
    id_resposta_amostra, id_pergunta_amostra = cur.fetchone()

    cur.execute("""
        SELECT id_modelo
        FROM modelos
        WHERE tipo_modelo IN ('juiz', 'ambos')
        ORDER BY id_modelo
        LIMIT 1;
    """)
    row_modelo_juiz = cur.fetchone()

    if row_modelo_juiz:
        id_modelo_juiz_amostra = row_modelo_juiz[0]
    else:
        cur.execute("""
            INSERT INTO modelos (nome_modelo, versao, parametro_precisao, tipo_modelo)
            VALUES ('Juiz Amostra 2+1', 'validacao', 'n/a', 'juiz')
            RETURNING id_modelo;
        """)
        id_modelo_juiz_amostra = cur.fetchone()[0]

    cur.execute("""
        INSERT INTO respostas_auxiliares_juiz (
            id_pergunta, id_modelo_juiz, texto_resposta, objetivo,
            prompt_utilizado, versao_prompt, parametros_geracao,
            tempo_inferencia_ms, finish_reason, observacoes
        ) VALUES (%s, %s, %s, 'auditoria', %s, %s, %s, 10, 'stop', %s)
        RETURNING id_resposta_auxiliar;
    """, (
        id_pergunta_amostra,
        id_modelo_juiz_amostra,
        'Resposta auxiliar transacional para auditoria; nao e gabarito.',
        'Prompt auxiliar de validacao',
        'validacao-2-mais-1',
        Json({'temperature': 0}),
        Json({'audit_only': True}),
    ))
    id_resposta_auxiliar = cur.fetchone()[0]

    ids_avaliacao = {}
    for papel, rodada, nota, motivo in [
        ('principal', 'padrao', 3, None),
        ('controle', 'controle_qualidade', 5, 'Validacao transacional'),
        ('arbitro', 'arbitragem', 4, 'Validacao transacional'),
    ]:
        cur.execute("""
            INSERT INTO avaliacoes_juiz (
                id_resposta_ativa1, id_modelo_juiz, nota_atribuida,
                prompt_juiz, rubrica_utilizada, chain_of_thought,
                papel_juiz, rodada_julgamento, motivo_acionamento,
                id_resposta_auxiliar
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            RETURNING id_avaliacao;
        """, (
            id_resposta_amostra,
            id_modelo_juiz_amostra,
            nota,
            'Prompt de validacao 2+1',
            'Rubrica registrada em avaliacoes_juiz',
            'Justificativa auditavel de validacao',
            papel,
            rodada,
            motivo,
            id_resposta_auxiliar,
        ))
        ids_avaliacao[papel] = cur.fetchone()[0]

    cur.execute("""
        INSERT INTO avaliacao_erros (
            id_avaliacao, tipo_erro, severidade, descricao,
            trecho_resposta, trecho_justificativa_juiz,
            impacto_na_nota, recomendado_ajuste_rubrica
        ) VALUES (%s, 'alucinacao_normativa', 'alta', 'Erro transacional de validacao', NULL, NULL, -1, false);
    """, (ids_avaliacao['controle'],))

    cur.execute("""
        INSERT INTO human_audits (
            id_avaliacao, auditor_nome, concorda_com_juiz,
            nota_humana, comentario, erro_do_juiz_detectado, tipo_erro_juiz
        ) VALUES (%s, 'Auditor Validacao', false, 4, 'Divergencia transacional', true, 'justificativa_inconsistente');
    """, (ids_avaliacao['controle'],))

    cur.execute("""
        INSERT INTO decisoes_finais_julgamento (
            id_resposta_ativa1, nota_final, metodo_decisao,
            id_avaliacao_principal, id_avaliacao_controle, id_avaliacao_arbitro,
            divergencia_detectada, diferenca_notas, arbitragem_acionada,
            motivo_arbitragem, justificativa_decisao
        ) VALUES (%s, 4, 'decisao_arbitro', %s, %s, %s, true, 2, true,
                  'Diferenca de notas >= 2', 'Arbitragem validada em transacao');
    """, (
        id_resposta_amostra,
        ids_avaliacao['principal'],
        ids_avaliacao['controle'],
        ids_avaliacao['arbitro'],
    ))

    cur.execute("""
        SELECT COUNT(*)
        FROM decisoes_finais_julgamento d
        JOIN avaliacoes_juiz ap ON d.id_avaliacao_principal = ap.id_avaliacao
        JOIN avaliacoes_juiz ac ON d.id_avaliacao_controle = ac.id_avaliacao
        JOIN avaliacoes_juiz aa ON d.id_avaliacao_arbitro = aa.id_avaliacao
        JOIN respostas_auxiliares_juiz raj ON ap.id_resposta_auxiliar = raj.id_resposta_auxiliar
        WHERE d.id_resposta_ativa1 = %s;
    """, (id_resposta_amostra,))
    total_validado = cur.fetchone()[0]

    if total_validado != 1:
        raise RuntimeError('Amostra transacional 2+1 nao fechou os relacionamentos esperados')

    conn.rollback()
    print('✅ Amostra transacional 2+1 validada e revertida com ROLLBACK')

except Exception:
    conn.rollback()
    raise


In [10]:
# =========================================
# 9. MODELO JUIZ — INSERT SEGURO SEM UPSERT
# =========================================
# ON CONFLICT (nome_modelo) exige UNIQUE constraint, que não existe no schema
# do professor. Usamos SELECT + INSERT condicional como alternativa segura.


# Sincroniza a sequência do ID com o valor máximo atual
cur.execute("SELECT setval('modelos_id_modelo_seq', (SELECT MAX(id_modelo) FROM modelos));")
conn.commit()

# Agora tente o seu código de INSERT novamente
cur.execute("SELECT id_modelo FROM modelos WHERE nome_modelo = 'M-Prometheus-7B';")
resultado = cur.fetchone()

if resultado:
    id_modelo_juiz = resultado[0]
    print("🧠 Modelo juiz já existe. ID:", id_modelo_juiz)
else:
    cur.execute("""
        INSERT INTO modelos (nome_modelo, versao, parametro_precisao, tipo_modelo)
        VALUES ('M-Prometheus-7B', 'v1.0', '4bit', 'juiz')
        RETURNING id_modelo;
    """)
    id_modelo_juiz = cur.fetchone()[0]
    conn.commit()
    print("🧠 Modelo juiz inserido com sucesso. ID:", id_modelo_juiz)


🧠 Modelo juiz inserido com sucesso. ID: 5


In [11]:
# =========================================
# 10. CARREGAR PROMETHEUS
# =========================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Unbabel/M-Prometheus-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

print("✅ Modelo carregado")

config.json:   0%|          | 0.00/790 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Modelo carregado


In [12]:
# =========================================
# 11. FUNÇÕES PROMETHEUS — DOMÍNIO JURÍDICO
# =========================================
import re


# Fonte: https://huggingface.co/Unbabel/M-Prometheus-7B
SISTEMA_JUIZ = (
    "Você é um Desembargador e Professor de Direito brasileiro, especialista em exames da OAB. "
    "Sua tarefa é fornecer feedbacks técnicos, imparciais e objetivos, avaliando as respostas "
    "estritamente de acordo com o gabarito oficial e a legislação brasileira vigente."
)


def montar_prompt(pergunta: str, resposta_modelo: str, gabarito: str, nome_dataset: str) -> str:
    is_objetiva = (nome_dataset == "OAB_Exames")

    if is_objetiva:
        rubrica = RUBRICA_OBJETIVA
        criterio = "A resposta da IA contém exatamente a alternativa correta do gabarito?"
        persona = (
            "Você é um Desembargador avaliando se uma IA acertou uma questão objetiva da OAB. "
            "Ignore qualquer explicação — apenas verifique se a letra final coincide com o gabarito."
        )
    else:
        rubrica = RUBRICA_DISCURSIVA
        criterio = "A resposta está juridicamente correta, bem fundamentada e alinhada ao gabarito oficial?"
        persona = (
            "Você é um Desembargador e Professor Doutor em Direito brasileiro. "
            "Sua tarefa é avaliar a resposta de uma IA a uma questão discursiva da OAB. "
            "Foque estritamente na precisão técnica e fundamentação legal."
        )

    return (
        "###Task Description:\n"
        f"{persona}\n\n"
        "Sua tarefa é fornecer um feedback detalhado e uma nota de 1 a 5 com base na rubrica fornecida.\n"
        "1. Escreva um feedback detalhado (RACIOCÍNIO) em Português.\n"
        "2. O formato de saída DEVE seguir rigorosamente este padrão:\n"
        "RACIOCÍNIO: (seu feedback aqui) [RESULT] (nota de 1 a 5)\n"
        "3. Não adicione saudações, introduções ou conclusões extras.\n\n"

        f"###The instruction to evaluate:\n{pergunta}\n\n"
        f"###Response to evaluate:\n{resposta_modelo}\n\n"
        f"###Reference Answer (Score 5):\n{gabarito}\n\n"

        f"###Score Rubrics: [{criterio}]\n{rubrica}\n\n"

        "###Feedback:\n"
        "RACIOCÍNIO:"
    )



def montar_entrada(pergunta: str, resposta_modelo: str, gabarito: str, nome_dataset: str) -> str:
    """
    Aplica o template de chat do Mistral ao prompt do Prometheus.
    Obrigatório conforme documentação oficial — omitir causa comportamento inesperado.
    O Mistral não suporta role 'system' nativamente, por isso a mensagem de sistema
    é concatenada ao corpo no turno do usuário.
    """
    corpo = montar_prompt(pergunta, resposta_modelo, gabarito, nome_dataset)
    mensagens = [{"role": "user", "content": SISTEMA_JUIZ + "\n\n" + corpo}]
    return tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )


def extrair_nota(texto: str) -> int | None:
    """
    Extrai a nota numérica da saída do modelo.
    Prioridade ajustada para o padrão nativo do M-Prometheus.
    """
    # 1. Padrão Nativo M-Prometheus: "[RESULT] N" (O mais confiável)
    match = re.search(r"\[RESULT\]\s*([1-5])", texto)
    if match:
        return int(match.group(1))

    # 2. Padrão solicitado: "NOTA: N"
    match = re.search(r"NOTA[:\s]+([1-5])", texto, re.IGNORECASE)
    if match:
        return int(match.group(1))

    # 3. Fallback: "score: N"
    match = re.search(r"score[:\s]+([1-5])", texto, re.IGNORECASE)
    if match:
        return int(match.group(1))

    # 4. Último recurso: último dígito isolado entre 1 e 5
    # (Evita pegar números de artigos de lei no meio do texto)
    matches = re.findall(r"\b([1-5])\b", texto)
    if matches:
        return int(matches[-1])

    return None


def extrair_raciocinio(texto: str) -> str:
    """
    Extrai o trecho de raciocínio jurídico da saída do modelo,
    usando split("NOTA:") conforme recomendação do professor.
    Retorna o texto completo caso o separador não seja encontrado.
    """
    # Divide pelo separador oficial do modelo
    if "[RESULT]" in texto:
        return texto.split("[RESULT]")[0].replace("RACIOCÍNIO:", "").strip()
    # Fallback para o padrão anterior
    if "NOTA:" in texto.upper():
        return texto.upper().split("NOTA:")[0].replace("RACIOCÍNIO:", "").strip()
    return texto.strip()


def avaliar(pergunta: str, resposta_modelo: str, gabarito: str, nome_dataset: str) -> dict:
    """
    Executa a avaliação Prometheus para um par (resposta, gabarito).

    Retorna dict com:
        'prompt'      — prompt completo enviado (salvo em prompt_juiz no banco)
        'output'      — saída bruta gerada pelo modelo
        'raciocinio'  — chain-of-thought extraído (salvo em chain_of_thought)
        'nota'        — nota inteira de 1 a 5 (ou None se falhar)
    """
    prompt_completo = montar_entrada(pergunta, resposta_modelo, gabarito, nome_dataset)
    inputs = tokenizer(prompt_completo, return_tensors="pt", truncation=False).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    saida = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    return {"prompt": prompt_completo, "output": saida}


print("✅ Funções definidas — Juiz Jurídico (OAB) em Português")

✅ Funções definidas — Juiz Jurídico (OAB) em Português


In [13]:
# =========================================
# 12. LIMPAR AVALIAÇÕES ANTERIORES (OPCIONAL)
# =========================================
if REFAZER_AVALIACAO:
    cur.execute("""
        DELETE FROM avaliacoes_juiz
        WHERE id_modelo_juiz = %s
          AND (papel_juiz IS NULL OR papel_juiz = 'principal');
    """, (id_modelo_juiz,))
    conn.commit()
    print(f"🗑️ Avaliações do modelo {id_modelo_juiz} removidas — pipeline será refeito do zero")
else:
    print("⏭️ REFAZER_AVALIACAO = False — avaliações existentes preservadas")


🗑️ Avaliações do modelo 5 removidas — pipeline será refeito do zero


In [14]:
# =========================================
# 13. BUSCAR DADOS (APENAS NÃO AVALIADOS)
# =========================================
# Filtra respostas que ainda não possuem avaliação para o modelo juiz atual,
# evitando duplicatas caso o notebook seja re-executado.

query = """
SELECT
    r.id_resposta,
    p.enunciado,
    p.resposta_ouro,
    r.texto_resposta,
    d.nome_dataset
FROM respostas_atividade_1 r
JOIN perguntas p ON r.id_pergunta = p.id_pergunta
JOIN datasets d  ON p.id_dataset  = d.id_dataset
WHERE r.id_resposta NOT IN (
    SELECT id_resposta_ativa1
    FROM avaliacoes_juiz
    WHERE id_modelo_juiz = %s
      AND (papel_juiz IS NULL OR papel_juiz = 'principal')
)
AND d.nome_dataset IN ('OAB_Bench')
"""

if BATCH_LIMIT:
    query += f" LIMIT {int(BATCH_LIMIT)}"

cur.execute(query, (id_modelo_juiz,))
rows = cur.fetchall()
print(f"📊 Respostas pendentes de avaliação: {len(rows)}")


📊 Respostas pendentes de avaliação: 10


In [15]:
# =========================================
# 14. LOOP DE AVALIAÇÃO COM TRATAMENTO DE ERROS
# =========================================
sucessos = 0
falhas   = 0
sem_nota = 0

for id_resposta, enunciado, gabarito, resposta_modelo, nome_dataset in rows:
    print(f"\n{'='*60}")
    print(f"🔍 Avaliando ID {id_resposta} | Dataset: {nome_dataset}")

    try:
        avaliacao      = avaliar(enunciado, resposta_modelo, gabarito, nome_dataset)
        prompt_enviado = avaliacao["prompt"]
        resultado      = avaliacao["output"]
        print("🔎 Saída do modelo:\n", resultado[:600])

        nota = extrair_nota(resultado)
        print("📝 Nota extraída:", nota)

        if nota is None:
            print("⚠️  Não foi possível extrair nota — registro pulado")
            sem_nota += 1
            continue

        # Rubrica salva no banco também varia por dataset
        rubrica_salva = RUBRICA_OBJETIVA if nome_dataset == "OAB_Exames" else RUBRICA_DISCURSIVA

        cur.execute("""
            SELECT id_avaliacao FROM avaliacoes_juiz
            WHERE id_resposta_ativa1 = %s
              AND id_modelo_juiz = %s
              AND (papel_juiz IS NULL OR papel_juiz = 'principal');
        """, (id_resposta, id_modelo_juiz))

        if cur.fetchone():
            cur.execute("""
                UPDATE avaliacoes_juiz
                SET nota_atribuida     = %s,
                    prompt_juiz        = %s,
                    rubrica_utilizada  = %s,
                    chain_of_thought   = %s,
                    papel_juiz         = 'principal',
                    rodada_julgamento  = 'padrao'
                WHERE id_resposta_ativa1 = %s
                  AND id_modelo_juiz = %s
                  AND (papel_juiz IS NULL OR papel_juiz = 'principal');
            """, (
                nota, prompt_enviado, rubrica_salva, resultado,
                id_resposta, id_modelo_juiz,
            ))
            print("🔄 Registro atualizado")
        else:
            cur.execute("""
                INSERT INTO avaliacoes_juiz (
                    id_resposta_ativa1, id_modelo_juiz, nota_atribuida,
                    prompt_juiz, rubrica_utilizada, chain_of_thought,
                    papel_juiz, rodada_julgamentorompt,
                ) VALUES (%s, %s, %s, %s, %s, %s, 'principal', 'padrao');
            """, (
                id_resposta, id_modelo_juiz, nota,
                prompt_enviado, rubrica_salva, resultado,
            ))
            print("✅ Registro inserido")

        conn.commit()
        sucessos += 1

    except torch.cuda.OutOfMemoryError:
        conn.rollback()
        print(f"💥 OOM na resposta {id_resposta} — pulando")
        torch.cuda.empty_cache()
        falhas += 1

    except Exception as e:
        conn.rollback()
        print(f"❌ Erro inesperado na resposta {id_resposta}: {e}")
        falhas += 1

print(f"\n{'='*60}")
print(f"✅ Sucessos : {sucessos}")
print(f"🔄 Atualizados incluídos nos sucessos")
print(f"⚠️  Sem nota : {sem_nota}")
print(f"❌ Falhas   : {falhas}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🔍 Avaliando ID 394 | Dataset: OAB_Bench
🔎 Saída do modelo:
 A resposta apresenta múltiplos erros técnicos e gramaticais que comprometem completamente a validade jurídica da peça. Primeiramente, utiliza expressões incorretas como "editlaw" e "critierius", evidenciando desconhecimento básico de terminologia jurídica. Além disso, a estruturação da peça é inadequada, misturando elementos de diferentes tipos processuais e apresentando informações conflitantes sobre o número de vagas disponíveis.

A fundamentação jurídica é completamente falha, pois menciona leis inexistentes ("editlaw") e artigos fictícios, demonstrando total desconhecimento do ordenamento 
📝 Nota extraída: 1
✅ Registro inserido

🔍 Avaliando ID 382 | Dataset: OAB_Bench
🔎 Saída do modelo:
 Analisando a peça apresentada, observa-se uma série de falhas significativas que comprometem seu mérito jurídico:

1. Erro fundamental na escolha do instrumento processual: A peça utiliza recursos extraordinário e especial, quando o corre

In [19]:
# =========================================
# 15. VALIDAR RESULTADOS
# =========================================

cur.execute("""
    SELECT
        d.nome_dataset                                        AS base,
        m_cand.nome_modelo                                    AS candidato,
        m_juiz.nome_modelo                                    AS juiz,
        COUNT(a.id_avaliacao)                                 AS total_avaliado,
        ROUND(AVG(a.nota_atribuida), 2)                       AS media,
        MIN(a.nota_atribuida)                                 AS minima,
        MAX(a.nota_atribuida)                                 AS maxima,
        -- Taxa de acerto: apenas para datasets objetivos (nota 1 = errou, nota 5 = acertou)
        ROUND(
            SUM(CASE WHEN a.nota_atribuida = 5 THEN 1 ELSE 0 END) * 100.0
            / COUNT(a.id_avaliacao), 1
        )                                                     AS taxa_acerto_pct
    FROM avaliacoes_juiz a
    JOIN respostas_atividade_1 r  ON a.id_resposta_ativa1 = r.id_resposta
    JOIN perguntas p              ON r.id_pergunta         = p.id_pergunta
    JOIN datasets d               ON p.id_dataset          = d.id_dataset
    JOIN modelos m_cand           ON r.id_modelo           = m_cand.id_modelo
    JOIN modelos m_juiz           ON a.id_modelo_juiz      = m_juiz.id_modelo
    GROUP BY d.nome_dataset, m_cand.nome_modelo, m_juiz.nome_modelo
    ORDER BY d.nome_dataset, m_cand.nome_modelo;
""")

rows_result = cur.fetchall()

DATASETS_OBJETIVOS = {"OAB_Exames"}  # adicione aqui outros datasets objetivos se houver

print(f"\n{'='*100}")
print(f"{'BASE':<20} {'CANDIDATO':<25} {'JUIZ':<20} {'TOTAL':>6} {'MÉDIA':>7} {'MIN':>5} {'MAX':>5} {'ACERTO%':>9}")
print(f"{'-'*100}")

for base, candidato, juiz, total, media, minima, maxima, taxa_acerto in rows_result:
    # Exibe taxa de acerto só para objetivos; para discursivos exibe '-'
    acerto_str = f"{taxa_acerto}%" if base in DATASETS_OBJETIVOS else "  -"
    print(
        f"{base:<20} {candidato:<25} {juiz:<20} "
        f"{total:>6} {str(media):>7} {str(minima):>5} {str(maxima):>5} {acerto_str:>9}"
    )

print(f"{'='*100}")

#cur.close()
#conn.close()

print("\n🏆 PIPELINE COMPLETO FINALIZADO")


BASE                 CANDIDATO                 JUIZ                  TOTAL   MÉDIA   MIN   MAX   ACERTO%
----------------------------------------------------------------------------------------------------
OAB_Bench            gemma2                    M-Prometheus-7B           3    1.00     1     1         -
OAB_Bench            gemma2                    Prometheus-7B             2    3.50     2     5         -
OAB_Bench            llama321b                 M-Prometheus-7B           4    1.00     1     1         -
OAB_Bench            llama321b                 Prometheus-7B             4    1.25     1     2         -
OAB_Bench            llama323b                 M-Prometheus-7B           3    1.00     1     1         -
OAB_Bench            llama323b                 Prometheus-7B             2    4.00     3     5         -

🏆 PIPELINE COMPLETO FINALIZADO


In [ ]:
# =========================================
# 15A. CONSULTAS DE AUDITORIA 2+1 (ESCOPO MINIMO)
# =========================================
# Observacao metodologica:
# - prompt_juiz, rubrica_utilizada, chain_of_thought e nota_atribuida permanecem em avaliacoes_juiz.
# - respostas_auxiliares_juiz sao artefatos de auditoria, nunca gabarito.

AUDIT_QUERIES = {
    '0. fluxo_professor': """
        SELECT p.resposta_ouro, r.texto_resposta, a.nota_atribuida
        FROM avaliacoes_juiz a
        JOIN respostas_atividade_1 r ON a.id_resposta_ativa1 = r.id_resposta
        JOIN perguntas p ON r.id_pergunta = p.id_pergunta;
    """,
    '1. votos_principal_only': """
        SELECT
            p.resposta_ouro,
            r.texto_resposta,
            a.nota_atribuida
        FROM avaliacoes_juiz a
        JOIN respostas_atividade_1 r ON a.id_resposta_ativa1 = r.id_resposta
        JOIN perguntas p ON r.id_pergunta = p.id_pergunta
        WHERE a.papel_juiz = 'principal';
    """,
    '2. votos_individuais': """
        SELECT
            p.id_pergunta,
            m_candidato.nome_modelo AS modelo_candidato,
            m_juiz.nome_modelo AS modelo_juiz,
            a.papel_juiz,
            a.rodada_julgamento,
            a.nota_atribuida,
            a.chain_of_thought
        FROM avaliacoes_juiz a
        JOIN respostas_atividade_1 r ON a.id_resposta_ativa1 = r.id_resposta
        JOIN perguntas p ON r.id_pergunta = p.id_pergunta
        JOIN modelos m_candidato ON r.id_modelo = m_candidato.id_modelo
        JOIN modelos m_juiz ON a.id_modelo_juiz = m_juiz.id_modelo
        ORDER BY p.id_pergunta, m_candidato.nome_modelo, a.papel_juiz;
    """,
    '3. decisao_final_consolidada': """
        SELECT
            p.resposta_ouro,
            r.texto_resposta,
            d.nota_final AS nota_atribuida
        FROM decisoes_finais_julgamento d
        JOIN respostas_atividade_1 r ON d.id_resposta_ativa1 = r.id_resposta
        JOIN perguntas p ON r.id_pergunta = p.id_pergunta;
    """,
    '4. casos_arbitragem': """
        SELECT
            d.id_resposta_ativa1,
            d.diferenca_notas,
            d.arbitragem_acionada,
            d.motivo_arbitragem
        FROM decisoes_finais_julgamento d
        WHERE d.diferenca_notas >= 2 OR d.arbitragem_acionada = TRUE;
    """,
    '5. erros_por_tipo_severidade': """
        SELECT
            tipo_erro,
            severidade,
            COUNT(*) AS total
        FROM avaliacao_erros
        GROUP BY tipo_erro, severidade
        ORDER BY severidade, total DESC;
    """,
    '6. auditorias_humanas_divergentes': """
        SELECT
            h.id_auditoria,
            h.id_avaliacao,
            h.auditor_nome,
            h.concorda_com_juiz,
            h.nota_humana,
            a.nota_atribuida AS nota_juiz,
            h.tipo_erro_juiz,
            h.comentario
        FROM human_audits h
        JOIN avaliacoes_juiz a ON h.id_avaliacao = a.id_avaliacao
        WHERE h.concorda_com_juiz = FALSE
           OR h.erro_do_juiz_detectado = TRUE;
    """,
}

for nome_consulta, sql in AUDIT_QUERIES.items():
    cur.execute(sql)
    rows_auditoria = cur.fetchmany(5)
    print(f"
{nome_consulta} — primeiras {len(rows_auditoria)} linhas")
    for row in rows_auditoria:
        print(row)

print("
✅ Consultas de auditoria 2+1 executadas")


In [17]:
# =========================================
# 16. GERAR DUMP ATUALIZADO DO BANCO
# =========================================
from datetime import datetime

sysdate = datetime.now().strftime("%Y%m%d_%H%M%S")
nome_dump = f"backup_atividade_2_{sysdate}.sql"

!PGPASSWORD={PG_PASSWORD} pg_dump -h {PG_HOST} -U {PG_USER} -d {PG_DB} -f {nome_dump}

print(f"✅ Dump gerado: {nome_dump}")

✅ Dump gerado: backup_atividade_2_20260423_133606.sql


In [18]:
# =========================================
# 17. DOWNLOAD DO DUMP
# =========================================
from google.colab import files
files.download(nome_dump)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>